In [ ]:
!pip install torch transformers tqdm

In [ ]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")
from tqdm import tqdm
import subprocess
from transformers import (
    AutoModel, AutoTokenizer, LlamaModel, LlamaTokenizer, RobertaModel, RobertaTokenizer,
    BertModel, BertTokenizer
)
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
!mkdir -p /content/gemma_model
!tar -xvzf /content/drive/MyDrive/DDI/gemma-transformers-1.1-2b-it-v1.tar.gz -C /content/gemma_model

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
df_descriptions = pd.read_csv("/content/drive/MyDrive/VRSEC/IV Year/Mini 2/Sahith/Datasets/Edited Dataset/Descriptions.csv")
df_descriptions.drop(index=2464, inplace=True)
df_descriptions

,DrugID,Descriptions
0,D00001,Polyethylene glycol is a laxative used to trea...
1,D00003,Activated charcoal is a gastric decontaminatio...
2,D00004,Coumarin is an anticoagulant that inhibits the...
3,D00006,Bupropion is a norepinephrine and dopamine reu...
4,D00008,Lindane is an ectoparasiticide and ovicide use...
...,...,...
2459,H01150,Abelmoschus moschatus [Syn. Hibiscus abelmosch...
2460,H01151,Asparagus falcatus is a medicinal plant tradit...
2461,H01152,Barleria prionitis is valued in herbal medicin...
2462,H01154,Solanum xanthocarpum is valued in herbal medic...


In [ ]:
def load_dataset(dataset_name, split=False):
    loader_function = getattr(dc.molnet, f'load_{dataset_name}', None)
    if loader_function is None:
        raise ValueError(f"Dataset {dataset_name} not found in DeepChem.")

    featurizer = dc.feat.DummyFeaturizer()

    if split:
        if dataset_name in ['bbbp', 'bace_classification', 'hiv']:
            splitter = dc.splits.ScaffoldSplitter()
        else:
            splitter = dc.splits.RandomSplitter()
        data = loader_function(featurizer=featurizer, splitter=splitter)
        tasks, (train_dataset, valid_dataset, test_dataset), transformers = data
        train_df = convert_to_dataframe(train_dataset, tasks)
        valid_df = convert_to_dataframe(valid_dataset, tasks)
        test_df = convert_to_dataframe(test_dataset, tasks)
        return train_df, valid_df, test_df
    else:
        data = loader_function(featurizer=featurizer, splitter=None)
        tasks, (dataset,), transformers = data
        df = convert_to_dataframe(dataset, tasks)
        return df

def convert_to_dataframe(dataset, tasks):
    df = pd.DataFrame(dataset.X, columns=['SMILES'])
    for i, task in enumerate(tasks):
        df[task] = dataset.y[:, i]
    return df

In [ ]:
def load_emb(dataset, language_model, dataset_path=None, embedding_path=None):
    df = pd.read_csv(f"{dataset_path}{dataset}.csv")
    features_df = pd.read_csv(f"{embedding_path}{dataset}_{language_model}.csv")
    if 'Unnamed: 0' in features_df.columns:
        features_df = features_df.drop(columns=['Unnamed: 0'])
    targets = df.drop(columns=['Descriptions']).to_numpy()
    features = features_df.to_numpy()
    ids = df['Descriptions'].tolist()
    return features, targets

In [ ]:
errored_indices = []
all_embeddings = []

def login_to_huggingface(token):
    subprocess.run(['huggingface-cli', 'login', '--token', token], check=True)


def select_model(model_name, openai_api_key=None, huggingface_token=None):
    if model_name == 'SBERT':
        tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
        model = AutoModel.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
    elif model_name == 'LLaMA2':
        login_to_huggingface(huggingface_token)
        tokenizer = LlamaTokenizer.from_pretrained("meta-llama/Llama-2-7b-hf")
        model = LlamaModel.from_pretrained("meta-llama/Llama-2-7b-hf")
        tokenizer.pad_token = tokenizer.eos_token
    elif model_name == 'Molformer':
        tokenizer = AutoTokenizer.from_pretrained("ibm/MoLFormer-XL-both-10pct")
        model = AutoModel.from_pretrained("ibm/MoLFormer-XL-both-10pct")
    elif model_name == 'ChemBERTa':
        tokenizer = RobertaTokenizer.from_pretrained("DeepChem/ChemBERTa-10M-MLM")
        model = RobertaModel.from_pretrained("DeepChem/ChemBERTa-10M-MLM")
    elif model_name == 'BERT':
        tokenizer = BertTokenizer.from_pretrained("google-bert/bert-base-uncased")
        model = BertModel.from_pretrained("google-bert/bert-base-uncased")
    elif model_name == 'RoBERTa_ZINC':
        tokenizer = RobertaTokenizer.from_pretrained("entropy/roberta_zinc_480m")
        model = RobertaModel.from_pretrained("entropy/roberta_zinc_480m")
    elif model_name == 'RoBERTa':
        tokenizer = RobertaTokenizer.from_pretrained("FacebookAI/roberta-base")
        model = RobertaModel.from_pretrained("FacebookAI/roberta-base")
    elif model_name == 'SimCSE':
        tokenizer = AutoTokenizer.from_pretrained("princeton-nlp/sup-simcse-bert-base-uncased")
        model = AutoModel.from_pretrained("princeton-nlp/sup-simcse-bert-base-uncased")
    elif model_name == 'Gemma':
        model_path = "/content/gemma_model"
        tokenizer = AutoTokenizer.from_pretrained(model_path)
        model = AutoModelForCausalLM.from_pretrained(model_path)
    elif model_name == 'Mol2Vec':
        featurizer = dc.feat.Mol2VecFingerprint()
        return None, featurizer, None
    else:
        raise ValueError(f"Unknown model name: {model_name}")
    return tokenizer, model, None

def process_transformers(smiles_list, tokenizer, model):
    embeddings = []
    errored = []
    for i, smile in enumerate(tqdm(smiles_list, desc="Processing Transformer Descriptions")):
        try:
            inputs = tokenizer(smile, return_tensors="pt", padding='max_length', truncation=True, max_length=512)
            outputs = model(**inputs, output_hidden_states=True)
            hidden_states = outputs.hidden_states[-1]
            mask = inputs['attention_mask']
            emb = ((hidden_states * mask.unsqueeze(-1)).sum(1) / mask.sum(-1).unsqueeze(-1))
            embeddings.append(emb.detach().cpu().numpy())
        except Exception as e:
            errored.append(i)
            print(f"Error processing SMILES at index {i}: {smile}\n{e}")
    return embeddings, errored



def get_embeddings_dataframe(embeddings, df, errored_indices):
    emb_array = np.squeeze(embeddings)
    emb_df = pd.DataFrame(emb_array)
    clean_df = df.drop(index=errored_indices).reset_index(drop=True)
    return emb_df, clean_df


In [ ]:
model_name = 'BERT'  # or any valid model name
tokenizer, model, client_or_featurizer = select_model(model_name)

if model_name in ['SBERT', 'LLaMA2', 'Molformer', 'ChemBERTa', 'BERT', 'RoBERTa_ZINC', 'RoBERTa', 'SimCSE', 'Gemma']:
    embeddings, errors = process_transformers(df_descriptions['Descriptions'].tolist(), tokenizer, model)


embedding_df, clean_df = get_embeddings_dataframe(embeddings, df_descriptions, errors)

Processing Transformer Descriptions: 100%|██████████| 2464/2464 [1:08:32<00:00,  1.67s/it]


In [ ]:
embedding_df

,0,1,2,3,4,5,6,7,8,9,...,758,759,760,761,762,763,764,765,766,767
0,-0.075489,0.025556,0.005122,-0.161835,0.438182,-0.039849,-0.068861,0.708428,-0.127271,-0.162536,...,-0.389471,-0.030208,0.082312,-0.079561,0.198455,0.051408,-0.298593,-0.096791,-0.257559,0.208149
1,-0.227845,0.305086,-0.359634,-0.172296,0.456120,-0.446814,0.534515,0.428162,0.155629,-0.012421,...,-0.305014,-0.286213,-0.119018,-0.067405,0.203054,-0.297512,0.156769,-0.303290,-0.366958,0.187023
2,-0.311377,-0.124975,-0.026083,-0.087903,0.139749,-0.349122,0.237537,0.238961,0.206219,0.442180,...,-0.166696,-0.142402,-0.153655,0.309015,0.211731,-0.285461,-0.136272,-0.143552,-0.306378,0.249425
3,-0.428390,-0.127989,-0.122209,-0.291601,0.243301,-0.351993,0.468679,0.302723,-0.267706,0.383810,...,-0.292646,-0.251588,-0.141761,-0.125242,0.260195,-0.293000,-0.105556,-0.325616,-0.137962,-0.110835
4,-0.058673,0.049557,-0.060103,-0.209602,0.227383,-0.154789,0.173132,0.558843,0.074777,-0.162160,...,-0.276850,-0.298175,-0.050367,0.070996,0.030254,-0.150572,-0.302902,-0.118322,-0.174532,0.149733
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2459,0.075208,0.174237,0.105180,-0.208319,0.443053,-0.207423,0.283454,0.354890,0.121421,0.146545,...,-0.166132,-0.036630,-0.114718,-0.193086,0.009166,-0.250057,-0.047965,-0.092094,-0.251607,0.075538
2460,0.087380,0.128996,-0.037217,-0.170774,0.293160,-0.189123,0.417360,0.322805,0.160857,0.326422,...,-0.199985,-0.086081,-0.020023,-0.055012,-0.007702,-0.392340,0.005330,-0.149509,-0.381374,0.037542
2461,-0.031390,0.199577,0.030086,-0.116529,0.311596,-0.242505,0.269057,0.329240,0.184556,0.347665,...,-0.126345,-0.168433,-0.224860,-0.004271,0.000988,-0.521520,-0.015375,-0.162177,-0.240265,0.221699
2462,-0.034985,0.235759,0.025804,-0.112639,0.250754,-0.299030,0.305698,0.390217,0.326436,0.390478,...,-0.125469,-0.183564,-0.111219,-0.100575,0.055770,-0.481352,-0.022869,-0.148272,-0.213433,0.208325


In [ ]:
from google.colab import files
embedding_df.to_csv('BERT_Embeddings.csv', index=False)
files.download('BERT_Embeddings.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>